# 00 Data Preparation — Men Shoes Size 8 (Mar 2025 – Mar 2026)

**Input :** `demand_modeling_data_8_v2/amazon_shoes_8_combined.parquet`

**Output:**
```
code/
    main_train_keys.csv
    main_val_keys.csv
  data/amzn_shoes_monthly_diffs_ffill_fixed_splits/
    train-00000-of-00001.parquet
    validation-00000-of-00001.parquet
```

**Pipeline order (senior DS standard):**
1. Load → remove dirty ASINs → filter Men
2. Drop dead columns → fix dtypes → rebuild subcat
3. **Forward fill on FULL data (Jan–Mar)** — Jan/Feb acts as seed for March
4. Create NaN flags on trimmed window only
5. Trim to Mar 2025 onward
6. Drop incomplete ASINs (entire ASIN, not rows) — show list first
7. Final checks → split → save → removal log

## ① Setup

In [1]:
# Dependencies: pyarrow, pandas (install via pip/conda if needed)
print('✅ Setup ready')

✅ Setup ready


## ② Config

In [2]:
import os, shutil
import pandas as pd
import numpy as np
import random
from pathlib import Path

# Detect project root by walking up from CWD to find 'data/' and 'code/' folders
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root (expected 'data/' and 'code/' folders)")

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_FILE = str(PROJECT_ROOT / 'data' / 'amazon_shoes_8_combined.parquet')
ROOT       = str(PROJECT_ROOT) + os.sep
PREP_DIR   = str(PROJECT_ROOT / 'data_preparation') + os.sep
DATA_DIR   = str(PROJECT_ROOT / 'data' / 'amzn_shoes_monthly_diffs_ffill_fixed_splits') + os.sep
CODE_DIR   = str(PROJECT_ROOT / 'code') + os.sep

os.makedirs(PREP_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

# ── Window ──────────────────────────────────────────────────────────────────
START_DATE = '2025-03-01'   # Analysis window start — Jan/Feb used as fill seed only
END_DATE   = '2026-03-31'

# ── Split ───────────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.5
SEED        = 42

# ── Columns ─────────────────────────────────────────────────────────────────
FFILL_COLS = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

DROP_COLS = [
    'SALES_RANK_original', 'PRICE_original', 'BUYBOX_PRICE_original',
    'gender', 'manufacturer', 'brand', 'model', 'color', 'size',
]

REQUIRED_COMPLETE = ['SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'RATING', 'REVIEW_COUNT']

NAN_TOLERANCE = 0.0   # strict — matches Option B

# ── Paper schema ────────────────────────────────────────────────────────────
PAPER_SCHEMA = {
    'ASIN': 'object', 'window': 'float64', 'date': 'datetime64[ns]',
    'SALES_RANK': 'float64', 'PRICE': 'float64', 'BUYBOX_PRICE': 'float64',
    'text': 'object', 'RATING': 'float64', 'REVIEW_COUNT': 'float64',
    'subcat': 'object', 'subcat_aggregated': 'object',
    'New Offer Count: Current': 'int64',
    'Count of retrieved live offers: New, FBA': 'int64',
    'Count of retrieved live offers: New, FBM': 'int64',
    'Lightning Deals: Upcoming Deal': 'int64',
    'Buy Box: Is FBA': 'int64',
    'image': 'object',
}

# ── Running removal log ──────────────────────────────────────────────────────
removal_log = []
def log_removal(asins, reason):
    for asin in asins:
        removal_log.append({'ASIN': asin, 'reason': reason})

print('✅ Config ready')
print(f'  Input       : {INPUT_FILE}')
print(f'  Output root : {ROOT}')
print(f'  Window      : {START_DATE} → {END_DATE}')
print(f'  Fill seed   : Jan + Feb 2025 (trimmed after fill)')
print(f'  Split       : {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)}')
print(f'  NaN tolerance: {NAN_TOLERANCE*100:.0f}% (0% = any NaN drops ASIN)')

✅ Config ready
  Input       : /home/iankuzuma/claude_code/demand-modeling-data-men-8-whole/data/amazon_shoes_8_combined.parquet
  Output root : /home/iankuzuma/claude_code/demand-modeling-data-men-8-whole/
  Window      : 2025-03-01 → 2026-03-31
  Fill seed   : Jan + Feb 2025 (trimmed after fill)
  Split       : 50/50
  NaN tolerance: 0% (0% = any NaN drops ASIN)


## ③ Load

In [3]:
df = pd.read_parquet(INPUT_FILE)
df['date'] = pd.to_datetime(df['date'])

print(f'Shape        : {df.shape}')
print(f'Unique ASINs : {df["ASIN"].nunique():,}')
print(f'Date range   : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Gender (ASIN level):')
print(df.drop_duplicates('ASIN')['gender'].value_counts().to_string())

Shape        : (1008930, 26)
Unique ASINs : 7,761
Date range   : 2025-01-06 → 2026-03-30

Gender (ASIN level):
gender
Women                                5472
Men                                  2279
Costumes & Accessories                  9
Shoe, Jewelry & Watch Accessories       1


## ④ Remove Dirty ASINs — Wrong Gender, Wrong Size, 100% NaN

In [4]:
# ── Wrong gender / wrong size ─────────────────────────────────────────────
BAD_GENDERS = ['Costumes & Accessories', 'Shoe, Jewelry & Watch Accessories']
gender_size_asins = df[
    df['gender'].isin(BAD_GENDERS) | (df['size'] != '8')
]['ASIN'].unique().tolist()

print(f'Wrong gender/size ASINs ({len(gender_size_asins)}):')
for asin in sorted(gender_size_asins):
    g = df[df['ASIN']==asin]['gender'].iloc[0]
    s = df[df['ASIN']==asin]['size'].iloc[0]
    print(f'  {asin}  gender={g}  size={s}')
log_removal(gender_size_asins, 'wrong gender or size')

# ── 100% NaN in any critical column (checked on Women only) ───────────────
CRITICAL_COLS = ['SALES_RANK', 'PRICE', 'RATING', 'REVIEW_COUNT']
df_men_temp = df[
    (df['gender'] == 'Men') & (~df['ASIN'].isin(gender_size_asins))
].copy()

print(f'\nChecking {df_men_temp["ASIN"].nunique():,} Men ASINs for 100% NaN...')
all_nan_asins = set()
for col in CRITICAL_COLS:
    fully_nan = df_men_temp.groupby('ASIN')[col].apply(lambda x: x.isna().all())
    flagged   = fully_nan[fully_nan].index.tolist()
    if flagged:
        print(f'  {col} — {len(flagged)} ASIN(s) 100% NaN:')
        for asin in sorted(flagged):
            subcat = df_men_temp[df_men_temp['ASIN']==asin]['subcat'].iloc[0]
            print(f'    {asin}  subcat={subcat}')
        all_nan_asins.update(flagged)
    else:
        print(f'  {col} — no 100% NaN ASINs ✅')
log_removal(list(all_nan_asins), '100% NaN in critical column')

dirty_asins = list(set(gender_size_asins) | all_nan_asins)
print(f'\nTotal dirty ASINs to remove: {len(dirty_asins)}')

Wrong gender/size ASINs (11):
  B0006NVZ28  gender=Costumes & Accessories  size=8


  B06XPBBSNX  gender=Costumes & Accessories  size=8
  B085Q368Y1  gender=Costumes & Accessories  size=8


  B092QKJXXT  gender=Costumes & Accessories  size=8
  B097MBKTL2  gender=Costumes & Accessories  size=8


  B09GFLPSX4  gender=Costumes & Accessories  size=8
  B09GFLT51Y  gender=Costumes & Accessories  size=8


  B09TDJCKD6  gender=Shoe, Jewelry & Watch Accessories  size=8
  B0C1ZB649F  gender=Costumes & Accessories  size=8
  B0D7CDDP5Q  gender=Costumes & Accessories  size=8


  B0DM7J3W65  gender=Men  size=7



Checking 2,278 Men ASINs for 100% NaN...
  SALES_RANK — no 100% NaN ASINs ✅
  PRICE — no 100% NaN ASINs ✅


  RATING — no 100% NaN ASINs ✅
  REVIEW_COUNT — no 100% NaN ASINs ✅

Total dirty ASINs to remove: 11


## ⑤ Filter to Men Only

In [5]:
before = df['ASIN'].nunique()
df = df[(df['gender'] == 'Men') & (~df['ASIN'].isin(dirty_asins))].copy()
print(f'ASINs : {before:,} → {df["ASIN"].nunique():,}  (kept Men, removed dirty)')
print(f'Rows  : {len(df):,}')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print()
print('Window distribution:')
print(df['window'].value_counts().to_string())

ASINs : 7,761 → 2,278  (kept Men, removed dirty)
Rows  : 296,140
Date range : 2025-01-06 → 2026-03-30

Window distribution:
window
7     148070
28    148070


## ⑥ Drop Dead Columns — Early to Reduce Memory

In [6]:
cols_dropped = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=cols_dropped)
print(f'Dropped {len(cols_dropped)} columns: {cols_dropped}')
print(f'Remaining ({len(df.columns)}): {df.columns.tolist()}')

Dropped 9 columns: ['SALES_RANK_original', 'PRICE_original', 'BUYBOX_PRICE_original', 'gender', 'manufacturer', 'brand', 'model', 'color', 'size']
Remaining (17): ['ASIN', 'window', 'date', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'RATING', 'REVIEW_COUNT', 'subcat', 'subcat_aggregated', 'New Offer Count: Current', 'Count of retrieved live offers: New, FBA', 'Count of retrieved live offers: New, FBM', 'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA', 'image']


## ⑦ Fix Dtypes

In [7]:
df['window'] = df['window'].astype('float64')
df['date']   = pd.to_datetime(df['date'])

# FBA and FBM — fill NaN with 0
# Keepa did not return offer data = treat as 0 active offers that week
for col in ['Count of retrieved live offers: New, FBA',
            'Count of retrieved live offers: New, FBM']:
    if col in df.columns:
        n = df[col].isna().sum()
        df[col] = df[col].fillna(0).astype('int64')
        short = col.split(': ')[-1]
        print(f'  {short:<8} filled {n:,} NaN → 0  (no Keepa offer data = 0 offers)')

df['New Offer Count: Current']       = df['New Offer Count: Current'].fillna(0).astype('int64')
df['Lightning Deals: Upcoming Deal'] = df['Lightning Deals: Upcoming Deal'].astype('int64')
df['Buy Box: Is FBA']                = df['Buy Box: Is FBA'].astype('int64')
print('✅ Dtypes fixed')

  New, FBA filled 263,285 NaN → 0  (no Keepa offer data = 0 offers)
  New, FBM filled 288,406 NaN → 0  (no Keepa offer data = 0 offers)
✅ Dtypes fixed


## ⑧ Rebuild subcat_aggregated — Top 10 Men Subcats + Other

In [8]:
# Compute Top 10 subcategories dynamically from the data
top_10_counts = (
    df.drop_duplicates('ASIN')
    .groupby('subcat')['ASIN'].count()
    .sort_values(ascending=False)
    .head(10)
)
TOP_10 = set(top_10_counts.index.tolist())
print(f'Top 10 subcategories (by ASIN count):')
for sc in top_10_counts.index:
    print(f'  {sc}: {top_10_counts[sc]}')

df['subcat_aggregated'] = df['subcat'].apply(lambda x: x if x in TOP_10 else 'Other')

result = (
    df.drop_duplicates('ASIN')
    .groupby('subcat_aggregated')['ASIN'].count()
    .sort_values(ascending=False).reset_index()
    .rename(columns={'ASIN': 'asin_count'})
)
result['pct'] = (result['asin_count'] / result['asin_count'].sum() * 100).round(1)
print(result.to_string(index=False))
print(f'\nTotal: {result["asin_count"].sum():,} ASINs | {result.shape[0]} groups')

Top 10 subcategories (by ASIN count):
  Loafers & Slip-Ons: 396
  Fashion Sneakers: 384
  Oxfords: 269
  Walking: 229
  Road Running: 204
  Trail Running: 98
  Water Shoes: 92
  Golf: 75
  Shoes: 69
  Track & Field & Cross Country: 61
            subcat_aggregated  asin_count  pct
                        Other         401 17.6
           Loafers & Slip-Ons         396 17.4
             Fashion Sneakers         384 16.9
                      Oxfords         269 11.8
                      Walking         229 10.1
                 Road Running         204  9.0
                Trail Running          98  4.3
                  Water Shoes          92  4.0
                         Golf          75  3.3
                        Shoes          69  3.0
Track & Field & Cross Country          61  2.7

Total: 2,278 ASINs | 11 groups


## ⑨ Fill Missing Values — On Full Data Including Jan/Feb

**This is the critical step.**
Jan and Feb 2025 act as the seed for the March window.
We fill NOW, before trimming, so March inherits valid values from earlier months.
BUYBOX_PRICE also fills any leading gaps using the first known price in the window.

In [9]:
df = df.sort_values(['ASIN', 'window', 'date']).reset_index(drop=True)
fill_cols = [c for c in FFILL_COLS if c in df.columns]

print('NaN counts BEFORE fill (full Jan–Mar data):')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

# Fill all columns forward per (ASIN, window)
# BUYBOX_PRICE additionally fills leading NaN using the first known value
# — products that gained a Buy Box during the window had no prior price data
other_cols  = [c for c in fill_cols if c != 'BUYBOX_PRICE']
buybox_cols = ['BUYBOX_PRICE'] if 'BUYBOX_PRICE' in fill_cols else []

if other_cols:
    df[other_cols] = (
        df.groupby(['ASIN', 'window'])[other_cols]
        .transform(lambda s: s.ffill())
    )

if buybox_cols:
    df['BUYBOX_PRICE'] = (
        df.groupby(['ASIN', 'window'])['BUYBOX_PRICE']
        .transform(lambda s: s.ffill().bfill())
    )

print('\nNaN counts AFTER fill:')
for col in fill_cols:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  ({n/len(df)*100:.2f}%)')

NaN counts BEFORE fill (full Jan–Mar data):
  SALES_RANK                          : 6,316  (2.13%)
  PRICE                               : 17,198  (5.81%)
  BUYBOX_PRICE                        : 70,397  (23.77%)
  RATING                              : 4,646  (1.57%)
  REVIEW_COUNT                        : 5,629  (1.90%)



NaN counts AFTER fill:
  SALES_RANK                          : 4,590  (1.55%)
  PRICE                               : 5,224  (1.76%)
  BUYBOX_PRICE                        : 7,280  (2.46%)
  RATING                              : 4,172  (1.41%)
  REVIEW_COUNT                        : 5,155  (1.74%)


## ⑩ Trim to Analysis Window, Then Create NaN Flags

Trim Jan/Feb NOW — after fill is done.
Create NaN flags on the trimmed window only, so they accurately reflect
which values were originally missing in the March→Mar 2026 period.

In [10]:
before_rows = len(df)
df = df[(df['date'] >= START_DATE) & (df['date'] <= END_DATE)].copy()

print(f'Rows  : {before_rows:,} → {len(df):,}  (removed {before_rows-len(df):,} Jan/Feb rows)')
print(f'Date range : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()

# NaN check on first date — should be near zero now
first_date = df['date'].min()
fw = df[df['date'] == first_date][fill_cols]
print(f'First date NaN check ({first_date.date()}) — should be near zero after fill:')
for col in fill_cols:
    n = fw[col].isna().sum()
    pct = n / len(fw) * 100
    flag = '✅' if pct < 1 else ('🟡' if pct < 10 else '🔴')
    print(f'  {col:<35} : {n:,} / {len(fw):,}  ({pct:.1f}%)  {flag}')

print()
# Create NaN flags on trimmed window
for col in fill_cols:
    if col in df.columns:
        df[f'is_{col}_filled'] = df[col].isna().astype(int)

print('NaN flag columns created on trimmed window:')
for col in fill_cols:
    flag = f'is_{col}_filled'
    n   = df[flag].sum()
    pct = n / len(df) * 100
    print(f'  {flag:<30} : {n:,} rows were NaN ({pct:.2f}%)')

Rows  : 296,140 → 259,692  (removed 36,448 Jan/Feb rows)
Date range : 2025-03-03 → 2026-03-30
ASINs : 2,278

First date NaN check (2025-03-03) — should be near zero after fill:
  SALES_RANK                          : 89 / 4,556  (2.0%)  🟡
  PRICE                               : 141 / 4,556  (3.1%)  🟡
  BUYBOX_PRICE                        : 112 / 4,556  (2.5%)  🟡
  RATING                              : 169 / 4,556  (3.7%)  🟡
  REVIEW_COUNT                        : 169 / 4,556  (3.7%)  🟡

NaN flag columns created on trimmed window:
  is_SALES_RANK_filled           : 1,070 rows were NaN (0.41%)
  is_PRICE_filled                : 2,778 rows were NaN (1.07%)
  is_BUYBOX_PRICE_filled         : 6,384 rows were NaN (2.46%)
  is_RATING_filled               : 1,938 rows were NaN (0.75%)
  is_REVIEW_COUNT_filled         : 1,938 rows were NaN (0.75%)


## ⑪ Find Incomplete ASINs — Show List Before Dropping

An ASIN is incomplete if any of its rows in the analysis window still have NaN
after fill. These cannot be fixed — the ASIN had no data to carry forward even
from Jan/Feb.

`NAN_TOLERANCE = 0.0` means strictly zero NaN allowed (any NaN = drop).
Change this in Config if you want to allow a small % of missing rows.

In [11]:
all_incomplete = {}

print('Checking for ASINs with remaining NaN after fill...')
print()

for col in REQUIRED_COMPLETE:
    # BUYBOX_PRICE: only drop if 100% missing — partial gaps were already filled
    # All other columns: use NAN_TOLERANCE (default 0 = any NaN = drop)
    threshold = 0.999 if col == 'BUYBOX_PRICE' else NAN_TOLERANCE

    nan_per_asin = (
        df[df['window'] == 7.0]
        .groupby('ASIN')[col]
        .apply(lambda x: (x.isna().sum() / len(x)) > threshold)
    )
    affected = nan_per_asin[nan_per_asin].index.tolist()
    all_incomplete[col] = affected

    if not affected:
        print(f'{col:<35} : no incomplete ASINs ✅')
    else:
        print(f'{col:<35} : {len(affected)} ASINs flagged:')
        for asin in sorted(affected)[:20]:
            subcat    = df[df['ASIN']==asin]['subcat'].iloc[0]
            nan_rows  = df[(df['ASIN']==asin) & df[col].isna()].shape[0]
            total_rows = df[df['ASIN']==asin].shape[0]
            pct_nan   = nan_rows / total_rows * 100
            print(f'  {asin}  subcat={subcat:<35}  NaN={nan_rows}/{total_rows} ({pct_nan:.0f}%)')
        if len(affected) > 20:
            print(f'  ... and {len(affected)-20} more')
    print()

asins_to_drop = set()
for asins in all_incomplete.values():
    asins_to_drop.update(asins)

print(f'Total unique ASINs to drop: {len(asins_to_drop):,}')
pct_lost = len(asins_to_drop) / df['ASIN'].nunique() * 100
print(f'That is {pct_lost:.1f}% of the dataset')
print()
if pct_lost > 10:
    print('⚠️  More than 10% of ASINs would be dropped — review the list above.')
else:
    print('✅ Within acceptable range — safe to drop.')

Checking for ASINs with remaining NaN after fill...

SALES_RANK                          : 53 ASINs flagged:
  B000MZFATO  subcat=Fashion Sneakers                     NaN=43/114 (38%)
  B07SZ1DMSN  subcat=Loafers & Slip-Ons                   NaN=105/114 (92%)
  B07VNYD2LG  subcat=Loafers & Slip-Ons                   NaN=17/114 (15%)


  B098F5865X  subcat=Fashion Sneakers                     NaN=2/114 (2%)
  B09JBZ8F1B  subcat=Walking                              NaN=8/114 (7%)


  B09MW198H4  subcat=Water Shoes                          NaN=9/114 (8%)
  B09VNVH87P  subcat=Fashion Sneakers                     NaN=99/114 (87%)
  B09W5GX29W  subcat=Fashion Sneakers                     NaN=105/114 (92%)
  B0B6WSD76J  subcat=Fashion Sneakers                     NaN=11/114 (10%)
  B0C5QHZWXX  subcat=Road Running                         NaN=26/114 (23%)
  B0C62CZ47J  subcat=Golf                                 NaN=19/114 (17%)
  B0C6T43K7D  subcat=Fashion Sneakers                     NaN=8/114 (7%)


  B0C932F4JP  subcat=Road Running                         NaN=1/114 (1%)
  B0CKRZM9PN  subcat=Road Running                         NaN=41/114 (36%)


  B0CNV3CCWZ  subcat=Fashion Sneakers                     NaN=10/114 (9%)
  B0CNV3M1M8  subcat=Fashion Sneakers                     NaN=10/114 (9%)
  B0CP6876Q1  subcat=Fashion Sneakers                     NaN=10/114 (9%)
  B0CRVFZL14  subcat=Fashion Sneakers                     NaN=33/114 (29%)
  B0D263CFX6  subcat=Basketball                           NaN=11/114 (10%)
  B0D2644K3M  subcat=Basketball                           NaN=11/114 (10%)
  ... and 33 more



...

  B0D1YF734M  subcat=Fashion Sneakers                     NaN=24/114 (21%)
  B0D263CFX6  subcat=Basketball                           NaN=11/114 (10%)
  ... and 77 more

REVIEW_COUNT                        : 97 ASINs flagged:
  B01N33VC2J  subcat=Fashion Sneakers                     NaN=9/114 (8%)
  B0B2F48TSZ  subcat=Road Running                         NaN=10/114 (9%)
  B0BTMWK5LF  subcat=Golf                                 NaN=1/114 (1%)


  B0BWRXT3P9  subcat=Basketball                           NaN=48/114 (42%)


  B0C5QHZWXX  subcat=Road Running                         NaN=26/114 (23%)
  B0C62CZ47J  subcat=Golf                                 NaN=19/114 (17%)
  B0CKRZM9PN  subcat=Road Running                         NaN=41/114 (36%)
  B0CNX8CJZ7  subcat=Rain                                 NaN=11/114 (10%)
  B0CPQ7MH9F  subcat=Fashion Sneakers                     NaN=19/114 (17%)
  B0CPQ8XHMJ  subcat=Fashion Sneakers                     NaN=19/114 (17%)
  B0CQF31LBZ  subcat=Road Running                         NaN=11/114 (10%)
  B0CRLD2Z81  subcat=Boots                                NaN=1/114 (1%)


  B0CRVFZL14  subcat=Fashion Sneakers                     NaN=34/114 (30%)


  B0CWS8CKH8  subcat=Fitness & Cross-Training             NaN=1/114 (1%)
  B0CWS9779Q  subcat=Fitness & Cross-Training             NaN=49/114 (43%)
  B0CYR2Z2P1  subcat=Fashion Sneakers                     NaN=75/114 (66%)
  B0CZHNJ73H  subcat=Road Running                         NaN=10/114 (9%)
  B0D17WQJG2  subcat=Fashion Sneakers                     NaN=10/114 (9%)
  B0D1YF734M  subcat=Fashion Sneakers                     NaN=24/114 (21%)
  B0D263CFX6  subcat=Basketball                           NaN=11/114 (10%)
  ... and 77 more

Total unique ASINs to drop: 217
That is 9.5% of the dataset

✅ Within acceptable range — safe to drop.


## ⑫ Execute Drop

In [12]:
for col, asins in all_incomplete.items():
    if asins:
        log_removal(asins, f'incomplete time series — NaN in {col} after ffill')

before_asins = df['ASIN'].nunique()
before_rows  = len(df)
df = df[~df['ASIN'].isin(asins_to_drop)].copy()

print(f'ASINs : {before_asins:,} → {df["ASIN"].nunique():,}  (dropped {before_asins - df["ASIN"].nunique():,})')
print(f'Rows  : {before_rows:,} → {len(df):,}  (dropped {before_rows - len(df):,})')
print()
print('NaN check after drop:')
for col in REQUIRED_COMPLETE:
    n = df[col].isna().sum()
    print(f'  {col:<35} : {n:,}  {"✅" if n==0 else "⚠️ "}')

ASINs : 2,278 → 2,061  (dropped 217)
Rows  : 259,692 → 234,954  (dropped 24,738)

NaN check after drop:
  SALES_RANK                          : 0  ✅
  PRICE                               : 0  ✅
  BUYBOX_PRICE                        : 0  ✅
  RATING                              : 0  ✅
  REVIEW_COUNT                        : 0  ✅


## ⑬ Reorder Columns — Paper Schema

In [13]:
CORE_COLS = [
    'ASIN', 'window', 'date',
    'SALES_RANK', 'is_SALES_RANK_filled',
    'PRICE', 'is_PRICE_filled',
    'BUYBOX_PRICE', 'is_BUYBOX_PRICE_filled',
    'text',
    'RATING', 'is_RATING_filled',
    'REVIEW_COUNT', 'is_REVIEW_COUNT_filled',
    'subcat', 'subcat_aggregated',
    'New Offer Count: Current',
    'Count of retrieved live offers: New, FBA',
    'Count of retrieved live offers: New, FBM',
    'Lightning Deals: Upcoming Deal',
    'Buy Box: Is FBA',
    'image',
]

missing = [c for c in CORE_COLS if c not in df.columns]
if missing:
    print(f'⚠️  Missing: {missing}')
else:
    df = df[CORE_COLS]
    print(f'✅ {len(df.columns)} columns in correct order')
    for i, col in enumerate(df.columns):
        print(f'  [{i:02d}] {col}')

✅ 22 columns in correct order
  [00] ASIN
  [01] window
  [02] date
  [03] SALES_RANK
  [04] is_SALES_RANK_filled
  [05] PRICE
  [06] is_PRICE_filled
  [07] BUYBOX_PRICE
  [08] is_BUYBOX_PRICE_filled
  [09] text
  [10] RATING
  [11] is_RATING_filled
  [12] REVIEW_COUNT
  [13] is_REVIEW_COUNT_filled
  [14] subcat
  [15] subcat_aggregated
  [16] New Offer Count: Current
  [17] Count of retrieved live offers: New, FBA
  [18] Count of retrieved live offers: New, FBM
  [19] Lightning Deals: Upcoming Deal
  [20] Buy Box: Is FBA
  [21] image


## ⑭ Schema Verification

In [14]:
print(f'{"#":<4} {"Column":<50} {"Expected":>16}  {"Actual":>16}')
print('-' * 94)
all_ok = True
for i, col in enumerate(df.columns):
    expected = PAPER_SCHEMA.get(col, 'object')
    actual   = str(df[col].dtype)
    # flag columns are int64 — expected is not in PAPER_SCHEMA, treat as OK
    ok = expected in actual or actual in expected or col.startswith('is_')
    mark = '✅' if ok else '❌'
    if not ok: all_ok = False
    print(f'{i:<4} {col:<50} {expected:>16}  {actual:>16}  {mark}')
print()
print('✅ All dtypes match!' if all_ok else '❌ Some dtypes do not match.')

#    Column                                                     Expected            Actual
----------------------------------------------------------------------------------------------
0    ASIN                                                         object            object  ✅
1    window                                                      float64           float64  ✅
2    date                                                 datetime64[ns]    datetime64[ns]  ✅
3    SALES_RANK                                                  float64           float64  ✅
4    is_SALES_RANK_filled                                         object             int64  ✅
5    PRICE                                                       float64           float64  ✅
6    is_PRICE_filled                                              object             int64  ✅
7    BUYBOX_PRICE                                                float64           float64  ✅
8    is_BUYBOX_PRICE_filled                                   

## ⑮ NaN Final Check

In [15]:
print(f'Shape : {df.shape}')
print(f'ASINs : {df["ASIN"].nunique():,}')
print()
nan_s = df.isna().sum().reset_index()
nan_s.columns = ['column', 'nan_count']
nan_s['nan_pct'] = (nan_s['nan_count'] / len(df) * 100).round(2)
nan_s['status']  = nan_s['nan_pct'].apply(
    lambda x: '🔴 HIGH' if x > 50 else ('🟡 MED' if x > 15 else '✅ OK')
)
print(nan_s.to_string(index=False))
remaining = nan_s[nan_s['nan_count'] > 0]
if len(remaining) == 0:
    print('\n✅ Zero NaN — all columns complete.')
else:
    print(f'\n⚠️  Remaining NaN in {len(remaining)} column(s) — review above.')

Shape : (234954, 22)
ASINs : 2,061

                                  column  nan_count  nan_pct status
                                    ASIN          0      0.0   ✅ OK
                                  window          0      0.0   ✅ OK
                                    date          0      0.0   ✅ OK
                              SALES_RANK          0      0.0   ✅ OK
                    is_SALES_RANK_filled          0      0.0   ✅ OK
                                   PRICE          0      0.0   ✅ OK
                         is_PRICE_filled          0      0.0   ✅ OK
                            BUYBOX_PRICE          0      0.0   ✅ OK
                  is_BUYBOX_PRICE_filled          0      0.0   ✅ OK
                                    text          0      0.0   ✅ OK
                                  RATING          0      0.0   ✅ OK
                        is_RATING_filled          0      0.0   ✅ OK
                            REVIEW_COUNT          0      0.0   ✅ OK
            

## ⑯ Stratified 50/50 ASIN-Level Split

In [16]:
rng = random.Random(SEED)
train_asins, val_asins = [], []
asin_subcat = df[['ASIN','subcat_aggregated']].drop_duplicates('ASIN')

print(f'Stratified {int(TRAIN_RATIO*100)}/{int((1-TRAIN_RATIO)*100)} split  seed={SEED}')
print(f'{"subcat_aggregated":<35} {"Total":>6}  {"Train":>8}  {"Val":>6}  {"Train%":>8}')
print('-' * 68)

for subcat, group in asin_subcat.groupby('subcat_aggregated'):
    asins   = sorted(group['ASIN'].unique().tolist())
    rng.shuffle(asins)
    n_train = max(1, round(len(asins) * TRAIN_RATIO))
    train_asins.extend(asins[:n_train])
    val_asins.extend(asins[n_train:])
    pct = n_train / len(asins) * 100
    print(f'  {subcat:<33} {len(asins):>6}  {n_train:>8}  {len(asins)-n_train:>6}  {pct:>7.1f}%')

print('-' * 68)
total = len(train_asins) + len(val_asins)
print(f'  {"TOTAL":<33} {total:>6}  {len(train_asins):>8}  {len(val_asins):>6}  {len(train_asins)/total*100:>7.1f}%')

overlap = set(train_asins) & set(val_asins)
assert len(overlap) == 0, f'ERROR: {len(overlap)} ASINs in both splits!'
print(f'\n✅ No overlap')

Stratified 50/50 split  seed=42
subcat_aggregated                    Total     Train     Val    Train%
--------------------------------------------------------------------
  Fashion Sneakers                     345       172     173     49.9%
  Golf                                  64        32      32     50.0%
  Loafers & Slip-Ons                   359       180     179     50.1%
  Other                                349       174     175     49.9%
  Oxfords                              261       130     131     49.8%
  Road Running                         171        86      85     50.3%
  Shoes                                 65        32      33     49.2%
  Track & Field & Cross Country         59        30      29     50.8%
  Trail Running                         82        41      41     50.0%
  Walking                              217       108     109     49.8%
  Water Shoes                           89        44      45     49.4%
-----------------------------------------------

## ⑰ Create Train and Val DataFrames

In [17]:
df_train = df[df['ASIN'].isin(set(train_asins))].copy()
df_val   = df[df['ASIN'].isin(set(val_asins))].copy()

print(f'Train : {len(df_train):,} rows  |  {df_train["ASIN"].nunique():,} ASINs')
print(f'Val   : {len(df_val):,} rows  |  {df_val["ASIN"].nunique():,} ASINs')
print(f'Same date range in both: {df_train["date"].min().date()} → {df_train["date"].max().date()}')

Train : 117,306 rows  |  1,029 ASINs
Val   : 117,648 rows  |  1,032 ASINs
Same date range in both: 2025-03-03 → 2026-03-30


## ⑱ Drop NaN Flag Columns Before Saving

In [18]:
flag_cols = [f'is_{c}_filled' for c in FFILL_COLS if f'is_{c}_filled' in df.columns]
df_train = df_train.drop(columns=flag_cols)
df_val   = df_val.drop(columns=flag_cols)
print(f'Dropped {len(flag_cols)} flag columns')
print(f'Final columns ({len(df_train.columns)}):')
for i, col in enumerate(df_train.columns):
    print(f'  [{i:02d}] {col:<50} {str(df_train[col].dtype)}')

Dropped 5 flag columns
Final columns (17):
  [00] ASIN                                               object
  [01] window                                             float64
  [02] date                                               datetime64[ns]
  [03] SALES_RANK                                         float64
  [04] PRICE                                              float64
  [05] BUYBOX_PRICE                                       float64
  [06] text                                               object
  [07] RATING                                             float64
  [08] REVIEW_COUNT                                       float64
  [09] subcat                                             object
  [10] subcat_aggregated                                  object
  [11] New Offer Count: Current                           int64
  [12] Count of retrieved live offers: New, FBA           int64
  [13] Count of retrieved live offers: New, FBM           int64
  [14] Lightning Deals: Upcoming Dea

## ⑲ Save Parquets and Key CSV Files

In [19]:
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

# Save parquets directly to data directory
train_path = os.path.join(DATA_DIR, 'train-00000-of-00001.parquet')
val_path   = os.path.join(DATA_DIR, 'validation-00000-of-00001.parquet')

df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)
print('✅ Parquets saved')
print(f'   train      : {len(df_train):,} rows')
print(f'   validation : {len(df_val):,} rows')

# Save key CSV files to code/ folder (used by 01_* notebooks)
train_keys_path = os.path.join(CODE_DIR, 'main_train_keys.csv')
val_keys_path   = os.path.join(CODE_DIR, 'main_val_keys.csv')

df_train[['ASIN','date']].assign(
    date=df_train['date'].dt.strftime('%Y-%m-%d')
).to_csv(train_keys_path, index=False)
df_val[['ASIN','date']].assign(
    date=df_val['date'].dt.strftime('%Y-%m-%d')
).to_csv(val_keys_path, index=False)
print('✅ Key CSV files saved to code/ folder')

✅ Parquets saved
   train      : 117,306 rows
   validation : 117,648 rows
✅ Key CSV files saved to code/ folder


## ⑳ Complete ASIN Removal Log

In [20]:
log_df = pd.DataFrame(removal_log)

print('=' * 65)
print('COMPLETE ASIN REMOVAL LOG')
print('=' * 65)
print(f'Total ASINs removed : {log_df["ASIN"].nunique():,}')
print(f'Total ASINs kept    : {df["ASIN"].nunique():,}')
print(f'Started with        : {log_df["ASIN"].nunique() + df["ASIN"].nunique():,} Men ASINs')
print()

# Summary by reason
summary = log_df.groupby('reason')['ASIN'].nunique().reset_index()
summary.columns = ['reason', 'count']
print('By reason:')
print(summary.to_string(index=False))
print()

# Detailed list by reason
for reason, group in log_df.groupby('reason'):
    asins = sorted(group['ASIN'].unique())
    print(f'── {reason}  ({len(asins)} ASINs) ──')
    for asin in asins:
        # Try to find subcat from the full dataset context
        rows = df_train[df_train['ASIN']==asin]
        if len(rows) == 0:
            rows = df_val[df_val['ASIN']==asin]
        subcat = rows['subcat'].iloc[0] if len(rows) > 0 else 'n/a (removed before split)'
        print(f'  {asin}  {subcat}')
    print()

COMPLETE ASIN REMOVAL LOG
Total ASINs removed : 228
Total ASINs kept    : 2,061
Started with        : 2,289 Men ASINs

By reason:
                                                  reason  count
incomplete time series — NaN in BUYBOX_PRICE after ffill     56
       incomplete time series — NaN in PRICE after ffill     80
      incomplete time series — NaN in RATING after ffill     97
incomplete time series — NaN in REVIEW_COUNT after ffill     97
  incomplete time series — NaN in SALES_RANK after ffill     53
                                    wrong gender or size     11

── incomplete time series — NaN in BUYBOX_PRICE after ffill  (56 ASINs) ──


  B0187Y1MNO  n/a (removed before split)
  B01N9VWWD8  n/a (removed before split)
  B07KBBNSHK  n/a (removed before split)
  B07PHCTYGL  n/a (removed before split)
  B083FK34PS  n/a (removed before split)
  B083FK37RQ  n/a (removed before split)
  B083FKBTMX  n/a (removed before split)
  B08B9P8MYH  n/a (removed before split)
  B08LGV5Q6R  n/a (removed before split)
  B08S93L6WD  n/a (removed before split)
  B09GKXK91L  n/a (removed before split)
  B09H3L9DW5  n/a (removed before split)
  B09JBZ8F1B  n/a (removed before split)
  B09NCWS2Q9  n/a (removed before split)
  B09NCWVQ88  n/a (removed before split)
  B09T9BLF31  n/a (removed before split)
  B09WC9P47T  n/a (removed before split)
  B09YH888W2  n/a (removed before split)
  B0B152VSYT  n/a (removed before split)
  B0B4MVP74L  n/a (removed before split)
  B0B57KSQTM  n/a (removed before split)
  B0B6Q54ZSQ  n/a (removed before split)
  B0BZZMD8C9  n/a (removed before split)
  B0C3YNH1PP  n/a (removed before split)
  B0C3YNYQ3C  n/

  B0C8J1QSTW  n/a (removed before split)
  B0CC4N33H4  n/a (removed before split)
  B0CCT7N3VH  n/a (removed before split)
  B0CFRD6YW7  n/a (removed before split)
  B0CFYNDCHS  n/a (removed before split)
  B0CFYNJJBN  n/a (removed before split)
  B0CN2MDZR4  n/a (removed before split)
  B0CN2VMV4H  n/a (removed before split)
  B0CN48QPTX  n/a (removed before split)
  B0CPDY1QR5  n/a (removed before split)
  B0CQ3W6MJV  n/a (removed before split)
  B0CQLVZYNQ  n/a (removed before split)
  B0CQSZNRHT  n/a (removed before split)
  B0CSJVCSQR  n/a (removed before split)
  B0CSJVJQND  n/a (removed before split)
  B0CSK27KZ5  n/a (removed before split)
  B0CWV18YW6  n/a (removed before split)
  B0CXH97QH6  n/a (removed before split)
  B0CXXK73QK  n/a (removed before split)
  B0CYR2Z2P1  n/a (removed before split)
  B0D5XH43J2  n/a (removed before split)
  B0D5XR8R1Z  n/a (removed before split)
  B0D5XT55DH  n/a (removed before split)
  B0D6VVMW68  n/a (removed before split)
  B0D7CPH87W  n/

  B0DP9MWZ81  n/a (removed before split)
  B0DP9PFZPW  n/a (removed before split)

── incomplete time series — NaN in PRICE after ffill  (80 ASINs) ──
  B000MZFATO  n/a (removed before split)
  B01HD6RPOC  n/a (removed before split)
  B01N9VWWD8  n/a (removed before split)
  B06XSMMLP7  n/a (removed before split)
  B079ZN5VX1  n/a (removed before split)
  B07MB7BNV5  n/a (removed before split)
  B07SZ1DMSN  n/a (removed before split)
  B07T391K1H  n/a (removed before split)


  B0844RCNN7  n/a (removed before split)
  B08C7BHJYP  n/a (removed before split)
  B08CZ3Z52Q  n/a (removed before split)
  B08KPW4WX5  n/a (removed before split)
  B08W9TM91W  n/a (removed before split)
  B08Z7MK41C  n/a (removed before split)
  B08ZHT9T8M  n/a (removed before split)
  B0916SFR4B  n/a (removed before split)
  B091KVLLCF  n/a (removed before split)
  B09GBFD2CF  n/a (removed before split)
  B09GKXHD91  n/a (removed before split)
  B09GLLGSDG  n/a (removed before split)
  B09GLLXTDC  n/a (removed before split)
  B09H3L9DW5  n/a (removed before split)
  B09J28XJ1X  n/a (removed before split)
  B09JBZ8F1B  n/a (removed before split)
  B09L4J3GCQ  n/a (removed before split)
  B09MF8DQKM  n/a (removed before split)
  B09MW198H4  n/a (removed before split)
  B09NCWS2Q9  n/a (removed before split)
  B09NCWVQ88  n/a (removed before split)
  B09PQZK4TF  n/a (removed before split)
  B09XCQ33WQ  n/a (removed before split)
  B0B14R43CB  n/a (removed before split)
  B0B2LN2Y2X  n/

...

  B0D657S1FT  n/a (removed before split)
  B0D65DVV4K  n/a (removed before split)
  B0D65LT5F2  n/a (removed before split)
  B0D677ZJ3D  n/a (removed before split)
  B0D68Y4KSV  n/a (removed before split)
  B0D68YML6M  n/a (removed before split)
  B0D68Z1K7R  n/a (removed before split)
  B0D6CSKVVS  n/a (removed before split)
  B0D6X7H6YB  n/a (removed before split)
  B0D7J1L4VR  n/a (removed before split)
  B0D8V6P9F1  n/a (removed before split)
  B0D95LGCC1  n/a (removed before split)
  B0D9PJ2WMV  n/a (removed before split)
  B0D9PJKDQ5  n/a (removed before split)
  B0D9V4S18L  n/a (removed before split)
  B0D9V5BKBQ  n/a (removed before split)
  B0D9V5FRSZ  n/a (removed before split)
  B0D9VZQNRZ  n/a (removed before split)
  B0DC71XC7X  n/a (removed before split)
  B0DFHJGMS8  n/a (removed before split)
  B0DFW8VWNX  n/a (removed before split)
  B0DGLYTPZG  n/a (removed before split)
  B0DH37P954  n/a (removed before split)
  B0DH38JHC7  n/a (removed before split)
  B0DH59LXFD  n/

  B0DK2H5LNG  n/a (removed before split)
  B0DKYS833Y  n/a (removed before split)
  B0DL24LDGH  n/a (removed before split)
  B0DLGWNKYD  n/a (removed before split)
  B0DLGWRCCR  n/a (removed before split)
  B0DLGZVNRV  n/a (removed before split)
  B0DLH38PB5  n/a (removed before split)
  B0DLMXS4GC  n/a (removed before split)
  B0DLTDLX4K  n/a (removed before split)
  B0DLTGLXVS  n/a (removed before split)


  B0DLY56PRP  n/a (removed before split)
  B0DLYN477V  n/a (removed before split)
  B0DM6YCRMM  n/a (removed before split)
  B0DM6YF6K7  n/a (removed before split)
  B0DM6YK7JH  n/a (removed before split)
  B0DN3GTSN5  n/a (removed before split)
  B0DPLPPKJ6  n/a (removed before split)
  B0DPLQB2R2  n/a (removed before split)
  B0DPLRWZL1  n/a (removed before split)
  B0DPRD51K9  n/a (removed before split)
  B0DPRDF89X  n/a (removed before split)
  B0DRWD49LT  n/a (removed before split)
  B0DRWFQPVC  n/a (removed before split)

── incomplete time series — NaN in SALES_RANK after ffill  (53 ASINs) ──
  B000MZFATO  n/a (removed before split)
  B07SZ1DMSN  n/a (removed before split)
  B07VNYD2LG  n/a (removed before split)
  B098F5865X  n/a (removed before split)
  B09JBZ8F1B  n/a (removed before split)
  B09MW198H4  n/a (removed before split)
  B09VNVH87P  n/a (removed before split)
  B09W5GX29W  n/a (removed before split)
  B0B6WSD76J  n/a (removed before split)
  B0C5QHZWXX  n/a (remov

  B0D9V5BKBQ  n/a (removed before split)
  B0D9VZ6QQD  n/a (removed before split)
  B0DH59LXFD  n/a (removed before split)
  B0DHLHJBXN  n/a (removed before split)
  B0DHYCPBQR  n/a (removed before split)
  B0DJ99JCWQ  n/a (removed before split)
  B0DJ9D3459  n/a (removed before split)
  B0DJL8MJS3  n/a (removed before split)
  B0DJLC7HPZ  n/a (removed before split)
  B0DJML3ZVR  n/a (removed before split)
  B0DJMLG84X  n/a (removed before split)
  B0DJMLWK7B  n/a (removed before split)
  B0DJMM8642  n/a (removed before split)
  B0DJTSDSBV  n/a (removed before split)
  B0DJTYB32Z  n/a (removed before split)
  B0DJWXQC6Z  n/a (removed before split)
  B0DK19MM4M  n/a (removed before split)
  B0DK2H5LNG  n/a (removed before split)
  B0DKYS833Y  n/a (removed before split)
  B0DLH38PB5  n/a (removed before split)
  B0DN3GTSN5  n/a (removed before split)
  B0DPLPPKJ6  n/a (removed before split)
  B0DPLQB2R2  n/a (removed before split)
  B0DPLRWZL1  n/a (removed before split)
  B0DRWFQPVC  n/

  B0DM7J3W65  n/a (removed before split)



## ㉑ Final Verification

In [21]:
df_tr = pd.read_parquet(DATA_DIR + 'train-00000-of-00001.parquet')
df_vl = pd.read_parquet(DATA_DIR + 'validation-00000-of-00001.parquet')

print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'Train  : {len(df_tr):,} rows  |  {df_tr["ASIN"].nunique():,} ASINs')
print(f'Val    : {len(df_vl):,} rows  |  {df_vl["ASIN"].nunique():,} ASINs')
print(f'Columns: {len(df_tr.columns)}')
print(f'Date   : {df_tr["date"].min()} → {df_tr["date"].max()}')
print()

overlap = set(df_tr['ASIN'].unique()) & set(df_vl['ASIN'].unique())
print(f'ASIN overlap : {len(overlap)}  {"✅" if len(overlap)==0 else "❌"}')
print()

print('subcat balance:')
tr_c = df_tr.drop_duplicates('ASIN')['subcat_aggregated'].value_counts().rename('train')
vl_c = df_vl.drop_duplicates('ASIN')['subcat_aggregated'].value_counts().rename('val')
bal  = pd.concat([tr_c, vl_c], axis=1).fillna(0).astype(int)
bal['total']   = bal['train'] + bal['val']
bal['train_%'] = (bal['train'] / bal['total'] * 100).round(1)
print(bal.sort_values('total', ascending=False).to_string())
print()

print('NaN check on saved files:')
all_clean = True
for col in ['SALES_RANK','PRICE','BUYBOX_PRICE','RATING','REVIEW_COUNT']:
    n_tr = df_tr[col].isna().sum()
    n_vl = df_vl[col].isna().sum()
    flag = '✅' if n_tr==0 and n_vl==0 else '⚠️ '
    if n_tr > 0 or n_vl > 0: all_clean = False
    print(f'  {col:<35} train={n_tr:,}  val={n_vl:,}  {flag}')

print()
if all_clean:
    print('✅ Data preparation complete — all columns clean!')
else:
    print('⚠️  Some NaN remain — review removal log and consider adjusting NAN_TOLERANCE')
print(f'\nNext: run image download notebook (~{df_tr["ASIN"].nunique()+df_vl["ASIN"].nunique():,} images)')

FINAL SUMMARY
Train  : 117,306 rows  |  1,029 ASINs
Val    : 117,648 rows  |  1,032 ASINs
Columns: 17
Date   : 2025-03-03 00:00:00 → 2026-03-30 00:00:00



ASIN overlap : 0  ✅

subcat balance:
                               train  val  total  train_%
subcat_aggregated                                        
Loafers & Slip-Ons               180  179    359     50.1
Other                            174  175    349     49.9
Fashion Sneakers                 172  173    345     49.9
Oxfords                          130  131    261     49.8
Walking                          108  109    217     49.8
Road Running                      86   85    171     50.3
Water Shoes                       44   45     89     49.4
Trail Running                     41   41     82     50.0
Shoes                             32   33     65     49.2
Golf                              32   32     64     50.0
Track & Field & Cross Country     30   29     59     50.8

NaN check on saved files:
  SALES_RANK                          train=0  val=0  ✅
  PRICE                               train=0  val=0  ✅
  BUYBOX_PRICE                        train=0  val=0  ✅
  RATING      